In [1]:
import os
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

In [2]:
cornwall_granular_collection = Chroma(
    collection_name="cornwall_granular",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)

In [3]:
cornwall_granular_collection.reset_collection()

In [4]:
cornwall_coarse_collection = Chroma(
    collection_name="cornwall_coarse",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)
cornwall_granular_collection.reset_collection()

In [5]:
os.environ["USER_AGENT"] = "manning-ch08/1.0 (mic.a.elle.chlon@gmail.com)"

from langchain_community.document_loaders import AsyncHtmlLoader
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()

Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  6.06it/s]


In [6]:
from langchain_text_splitters import HTMLSectionSplitter

header_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=header_to_split_on
)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(
            html_string
        )
        all_chunks.extend(temp_chunks)

    return all_chunks

In [7]:
granular_chunks = split_docs_into_granular_chunks(docs)

cornwall_granular_collection.add_documents(documents=granular_chunks)

['fcb753a6-9f81-4374-a188-29d35c7432bf',
 'ec64a7d8-fc22-4afd-9006-db983cdaaf59',
 '45d015f2-ad0b-4ec2-a77b-5c2e3405c8ea',
 'ec2ee6dd-29e4-4831-8449-1df9a6fdde3d',
 '408e23e7-601e-4dd9-9e10-02d34c0cb1b6',
 '6761f507-1825-4b4c-95c3-b1c1ef36735a',
 '149c031f-d522-4b5f-98d1-c7b673c372ef',
 '9b6237eb-b0f0-42e7-a1c9-65f507ba311c',
 '33352ae9-983a-4568-98c6-42873be0da32',
 '0378dcc5-be5c-4938-916c-5a17b0ba62ec',
 'ef0582cb-58bb-4659-abad-76ef1913ab86',
 'c1d5b3d0-4510-4f43-a208-24568014ddab',
 '1b4705a3-b526-4307-bb07-b4e7f9e2b538',
 '1074b3d6-5183-4ef1-8ef4-8395fd2d81c5',
 '913b1832-412b-4024-bdc6-4668e8b8bef2',
 'cebc264f-0454-4d96-9d9b-e7f0ed2932fc',
 '95e1e773-d966-46f2-8d59-5f8bedd09f7a',
 'c88e7e2a-b129-4ca6-bd8d-ee15d66cd0f8',
 '1880ce20-3994-4686-9575-cd6cf1ff0531']

In [8]:
results = cornwall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3
)

for doc in results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance 

In [9]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [10]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs)
    coarse_chunks = text_splitter.split_documents(text_docs)
    return coarse_chunks

In [11]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

cornwall_coarse_collection.add_documents(documents=coarse_chunks)

['11048d7d-7129-4efb-b949-94e6f22cecf9',
 'e42e74ae-45f2-4134-a386-71b5372ae550',
 '8f6d9b59-6807-4d59-a40f-18d3c0e46cbc',
 '9bc16a3b-573f-419e-8e76-49563b64db8a',
 '7aef268a-0c58-40cf-8e5b-8c0ce0d213a8',
 '40d02556-1181-474e-9df0-2bb52c86713f',
 '0ed51836-4af4-46b2-852e-c2692b5b451f',
 '4452c7d0-b70c-44af-8757-dc5f3fc6297f',
 '4d778e0b-099b-4144-bd45-67b3ae2e3262',
 '8bf76882-b255-44f7-891e-43bec32d51bc',
 '501415c2-e1b5-49be-87be-63118cfe6d07',
 'b26f207d-674a-4650-b7ee-166be069b876',
 '44f96633-8af7-48af-a593-55d75c8faa1a',
 '3712721e-110b-4b5f-862c-9b75981af57d',
 '749b36fc-3f19-4679-9410-c26853d9510a']

In [12]:
results = cornwall_coarse_collection.similarity_search(
    query="Events or Festival in Cornwall", k=3
)

for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corni

In [13]:
uk_granular_collection = Chroma(
    collection_name="uk_granular",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
uk_granular_collection.reset_collection()

uk_coarse_collection = Chroma(
    collection_name="uk_coarse",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
uk_coarse_collection.reset_collection()

uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' 
                       for d in uk_destinations]

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    docs = html_loader.load()

granular_chunks = split_docs_into_granular_chunks(docs)
uk_granular_collection.add_documents(documents=granular_chunks)

coarse_chunks = split_docs_into_coarse_chunks(docs)
uk_coarse_collection.add_documents(documents=coarse_chunks)

Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  6.85it/s]


['1e4ceba3-c468-4fb4-bb5d-248c4240528f',
 '31225e45-2774-4f60-9388-7ac6dab6e92d',
 '31239ad3-ba4f-4d45-853f-ee81597c546a',
 'b73b13ac-7751-42d7-a687-6a9efbcaa73f',
 'acd1fabb-df4b-4553-bc34-385538925864',
 '931db89d-020d-4400-9b00-ce6dc35a9cc4',
 '65d19ee7-2571-464d-b278-3602165bc304',
 '46db2e53-e9cc-4158-bd70-16b6100a197f',
 'abf0d020-caa4-48b0-ba8f-caa49a5edd19']

In [14]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)

for doc in granular_results:
    print(doc)
    print("\n------------------------------------------------------------------\n")

coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)

for doc in coarse_results:
    print(doc)

page_content='Go next 
 [ edit ] 
 
 
 
 Royal Tunbridge Wells  (on the A26) - Victorian spa town with bars, pubs and drinking fountains for the local water. 
 Eastbourne 
 Petersfield 
 
 South Downs Way , a popular walking path. 
 
 London , a train ride away. 
 Kent 
 Medway 
 Crowborough 
 
 
 
 
 Routes through Ashdown Forest 
 
 
 
 
 
 
 
 
 London   ←   East Grinstead   ← 
 
 
   N     S   
 
 
 →   Uckfield   →   Eastbourne 
 
 
 
 
 .mw-parser-output .routeBox{font-size:small;border-style:none;border-spacing:0 0;border-collapse:collapse;margin:0 auto}.mw-parser-output .routeBox td{padding:1px 2px} 
 
 
 
 
 
 
 
 .mw-parser-output .article-status{width:60%;background:#fff;color:black;margin:0 auto;border:solid 2px lightblue;text-align:center;font-size:90%;font-style:italic}.mw-parser-output .article-status-disambig{border:2px dashed lightblue}.mw-parser-output .article-status-disambig td:first-child{width:48px;text-align:center}.mw-parser-output .article-status-stub{border:1p

## 8.4/ Embedding strategy

In [15]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

child_chunk_collection = Chroma(
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY) 
)
child_chunk_collection.reset_collection()

doc_store = InMemoryStore()

parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=child_chunk_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [17]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)
    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None)

Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  5.30it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  5.88it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  7.67it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  5.83it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  5.72it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [18]:
list(doc_store.yield_keys())

['6b7677c2-73f7-45fa-8751-b3caee93d629',
 '74967ccc-a85e-485f-a43c-469960d704bb',
 'aebc2a86-eca8-4c5e-aff8-ecd30a35987c',
 '870a682a-9927-45ee-a200-e40974be153c',
 'ed6c4a22-bfdb-49fe-8b5f-b10424777ea0',
 '7423394d-89bb-45b5-af84-3811e5735333',
 '80238868-5390-444e-a687-1c51cea8fe5f',
 '53649d36-e2f5-4572-adad-d0dd9e9cbb14',
 '9b7a49ef-619e-4a44-8670-bf5c0ffd6c2b',
 'a0eda54f-d05f-4328-a50d-8478440655e2',
 '1561ba50-50b7-4898-96e3-ecaa2e3f9b05',
 '14b0dc12-6d88-497f-8130-1b19ff053221',
 'cc22b719-fd4f-486c-8ed6-b7b919200883',
 'f4c07a42-7d20-48d9-8d4f-61777d9b603d',
 '0bc8d39d-5c7e-4c51-9d89-c9f9106bfd64',
 'f2d7de67-39fe-4ceb-8dca-e6f01e28825c',
 'b7e4332e-0659-41b8-9310-2c67d421b775',
 '9a741f79-027e-4928-8421-0e2086f8a455',
 '6ea33e91-6fd3-4db2-ba11-e07440e3223f',
 'a1aa5d9b-8ab9-425f-adbf-329efd2c991a',
 'b3a0e78b-c463-4b1d-a5d8-33bfcca3e2d9',
 '9ffdf63b-6c5c-4a4a-85e3-455d82d3780d',
 'e4270c75-bc97-499a-84fe-53e40ce752d0',
 '9d16cd0b-81b3-4709-952f-4c2e403982d8',
 'b6173fe6-d14f-

In [19]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [20]:
print(retrieved_docs[0])

page_content='## Get around

[edit]

### By bus

[edit]

Thanks to Transport for Cornwall, all bus tickets are interchangeable across
the different companies. The **Cornwall All Day ticket** allows unlimited
travel for a calendar day. As of 2023, fares are £5 for adults and £4 for
under-19s. Payment is by cash or contactless. The two main bus companies are:

  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).
  * **Kernow** (part of First Bus) covers western and central Cornwall.

Buses only serve designated stops when in towns; otherwise, you can flag them
down anywhere that's safe for them to stop.

### By train

[edit]

**CrossCountry Trains** and **Great Western Railway** operate regular train
services between the main centres of population, the latter company also
serving a number of other towns on branch lines. For train times and fares
visit National Rail Enquiries.

The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall 

In [21]:
child_docs_only = child_chunk_collection.similarity_search("Cornwall Ranger")

In [22]:
print(child_docs_only)

[Document(id='f342fb1d-24de-47ed-aae9-7483e91ddadb', metadata={'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'doc_id': '9d16cd0b-81b3-4709-952f-4c2e403982d8'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.'), Document(id='7bd139b9-2ba5-4af3-bca1-53f9121c5c5b', metadata={'title': 'Cornwall – Travel guide at Wikivoyage', 'doc_id': '870a682a-9927-45ee-a200-e40974be153c', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='### Cornish\n\n[edit]'), Document(id='f6fbf4fd-4aa5-4c40-bc80-fefcfd115c6b', metadata={'title': 'Cornwall – Travel guide at Wikivoya

### 8.4.2/ Embedding child chunks with MultiVectorRetriever

In [23]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

In [24]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

child_chunks_collection = Chroma(
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY) 
)
child_chunks_collection.reset_collection()

doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)

In [25]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)

    coarse_chunks = parent_splitter.split_documents(text_docs)
    
    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(coarse_chunks):

        coarse_chunk_id = coarse_chunks_ids[i]
        granular_chunks = child_splitter.split_documents([coarse_chunk])

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id

        all_granular_chunks.extend(granular_chunks)
    
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(all_granular_chunks)
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  5.34it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  7.67it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  6.02it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  8.14it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  7.27it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [26]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
print(retrieved_docs)

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies. The **Cornwall All Day ticket** allows unlimited\ntravel for a calendar day. As of 2023, fares are £5 for adults and £4 for\nunder-19s. Payment is by cash or contactless. The two main bus companies are:\n\n  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).\n  * **Kernow** (part of First Bus) covers western and central Cornwall.\n\nBuses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\ns

In [27]:
child_docs_only = child_chunks_collection.similarity_search("Cornwall Ranger")
print(child_docs_only[0])

page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and
Plymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for
under-16s.

## See

[edit]

The **Eden Project** , near St Austell, a fabulous collection of flora from
all over the planet housed in two space age transparent domes, and a massive
zip line.' metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'doc_id': '0d3c9322-dcff-48a1-a36d-7f431f9a9ead', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


### 8.4.3/ Embedding document summaries 

In [28]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore

from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

In [29]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)

summaries_collection = Chroma(
    collection_name="uk_summaries",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)
summaries_collection.reset_collection()

In [30]:
doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

In [31]:
multi_vector_retriever = MultiVectorRetriever(
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)

In [32]:
llm = ChatOpenAI(model="gpt-5-nano", api_key=OPENAI_API_KEY)

summarization_chain = (
    {"document": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}")
    | llm
    | StrOutputParser()
)

In [33]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()

    text_docs = html2text_transformer.transform_documents(html_docs)

    coarse_chunks = parent_splitter.split_documents(text_docs)
    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(coarse_chunks):
        coarse_chunk_id = coarse_chunks_ids[i]
        summary_text = summarization_chain.invoke(coarse_chunk)
        summary_doc = Document(page_content=summary_text,
                               metadata={doc_key: coarse_chunk_id})
        all_summaries.append(summary_doc)
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(all_summaries)
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  5.32it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  6.91it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  6.20it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  7.81it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|############################################################################################################################################################| 1/1 [00:00<00:00,  7.21it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [35]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

summary_docs_only = summaries_collection.similarity_search("Cornwall Travel")
print(summary_docs_only[0])

page_content='- Overview: A concise travel guide for Cornwall, covering how to get there, get around, and notable sights.

Get in
- By train: Regular main-line trains run from London Paddington through Cornwall to Penzance (about 5 hours 30 minutes). An overnight sleeper operates on Sundays through Fridays from London Paddington and Penzance.
- By car: Access via the A30 from Exeter (expressway reaching Carland Cross near Truro by around March 2024) and via the A38 over the Tamar Bridge (toll for eastbound vehicles). Summer weekends and bank holidays can be busy.
- By plane: Cornwall Airport (NQY) in Newquay is the main airport. Year-round flights from Aberdeen, Alicante, Dublin, London Gatwick, and Manchester; more flights during the summer season.

Get around
- By bus: Transport for Cornwall allows interchange across operators. The Cornwall All Day ticket provides unlimited travel for a calendar day (about £5 for adults, £4 for under-19s as of 2023). Main operators are Go Cornwall Bu

### 8.4.4/ Embedding hypothetical questions